In [1]:
import pandas as pd
import numpy as np
from src.funct import sum_and_round, mean_and_round, plot_boxplot_with_points, mquintile, plot_q25


import matplotlib.pyplot as plt
import seaborn as sns

import altair as alt
from altair import datum #Needed for subsetting (transforming data)

In [2]:
boxplot_data = pd.read_pickle('./data/boxplot_data.pkl')
sales = boxplot_data[['Sub-Category', 'sales', 'type']]

sales_even = (sales
              .sort_values('sales')
              .reset_index(drop = True)
              .drop(int(sales.shape[0]/2))
              .reset_index(drop = True)
             )

print(sales.shape[0])
print(sales_even.shape[0])

17
16


## Quartile

Step-by-step

- Definition of Quantiles, Percentiles and Quartiles
- Computing quartiles for odd and even sized datasets 

Beside the median, `altair` boxplot shows the 1st and third quartile. Quartiles, like percentiles, are types of quantiles. The idea is to divide our dataset in $n$ equal parts, and using this concept to compare each data with the rest of the dataset. Usually, the answers a quintile is aimed to respond are "what is the position of this particular data with respect to the rest of the Dataset?", "how this outcome positions with respect to other possible outcomes?". A classic quantiles use case, are the results of an exam. If the obtained score is, for example, $80$, how high (or low) is it compared with the rest of the other scores? 

> TODO: Explain the difference between *percentile* and *percentile rank*.

In our example dataset, we have the sales for each category, and we want to know if Copiers sales are high or low with respect to the other categories. One way to evaluate this performance, is to see at which quantile it belongs to. 

In statistics, ***quantiles*** are particular points dividing a sample into equally sized, adjacent subgroups. The median is a quantiles so that exactly half of the data is lower than the median and half of the data is above the median: the median is said to be the 2<sup>nd</sup> quartile. Some type of quantiles have specific names like quartiles if we divide the distribution into four equal parts, quintiles, octiles, deciles if we divide respectively into 5, 8, 10 parts and percentiles if we divide into 100 equal parts. The most common quantiles are quartiles and percentiles (respectively 4 and 100 equal parts) that shares the following relationship:

- 0 quartile = 0 percentile (the minimum)
- 1st quartile =  25th percentile
- 2nd quartile = 50th percentile (the median)
- 3rd quartile = 75th percentile
- 4th quartile = 100th percentile (the maximum)

The first quartile is also known as Q1 or the lower quartile is the value of the 25<sup>th</sup> percentile; The third quartile is also known as Q3 or the upper quartile and is the value of the 75<sup>th</sup> percentile.

Although the notion of percentile and quartile are widely used in describing dataset, there is not a stardard definition for them. In fact, there are different methods for calculating percentiles and quantiles. Basically, there are three ways of considering if data belongs to a given quantile:

1. The smallest value that is greater than the $k$ percent.
2. The smallest value that is greater or equal to $k$ percent of values.
3. An interpolated value between the two closest rank

And we will also see, that analytically, there can be different ways to compute percentiles. In fact, if we look at the definition that takes into account the interpolated value betwee the two closest rank, we can think of different ways on how to interpolate between this two values. For our purposes, we will discuss two of the most widely used algorithm and, in the spirit of reinventing the wheel, we wil look at how Excel and Python compute the quantity using built in functions. 

Let's consider a small dataset consisting of 12 elements and go back to the definition of 1st quartile as 

1. the smallest value that is greater than the $25/%$ of the data
2. Definition 2: the smallest value that is greater or equal than the $25/%$ of the data
3. Definition 3: An interpolated value between the two closest rank

If we write the dataset as:

|           |   |   |   |   |   |   |   |   |   |    |    |    |
|:----------|:--|:--|:--|:--|:--|:--|:--|:--|:--|:---|:---|:---|
| **Rank**  | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 | 12 |
| **Value** | $a_{1}$ | $a_{2}$ | $a_{3}$ | $a_{4}$ | $a_{5}$ | $a_{6}$ | $a_{7}$ | $a_{8}$ | $a_{9}$ | $a_{10}$ | $a_{11}$ | $a_{12}$ |

we can brake it into 4 equal parts and the first quartile will be greater than the first three elements so that $1/4$ (i.e. $3/12$) of the data lies below this cut point. As a practical example, suppose the dataset, already sorted, is:

|           |   |   |   |   |   |   |   |   |   |    |    |    |
|:----------|--:|--:|--:|--:|--:|--:|--:|--:|--:|---:|---:|---:|
| **Rank**  | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 | 12 |
| **Value** | -78 | -56 | -44 | -22 | -5 | 12 | 32 | 45 | 52 | 61 | 68 | 78 |

the first quartile, i.e. the cut point dividing the dataset into the $25\%$ and $75\%$ of the data, must lie between the third and forth rank.

In [3]:
test_array = np.array([-78, -56, -44, -22, -5, 12, 32, 45, 52, 61, 68, 78])
test_data = pd.DataFrame({
    'x' : test_array, 
})

plot_q25(test_data, [-44, -22])

alt.LayerChart(...)

As we did it for the median, we can start by calculating which index is below the $25\%$ of the data. We can multiply the number of samples in the dataset by the percentage of samples we want to consider and, as a result, we will find the index representing the cut point that is equal to the quantity. If we use as a definition *greater than*, we can find the index by increasing by one the length of the array

In [4]:
percentage = 0.25
n_of_samples = len(test_array)
idx = percentage*(n_of_samples+1)
idx

3.25

The index we found is 3 and a quarter, meaning that the first quartile is between the 3<sup>rd</sup> and the 4<sup>th</sup> index. One way to compute the quartile when we have, as a result of computing the hypothetical index, a fractional number, is to compute the linear weighted mean of this middle point by using the following formula:

$$
q25 = lower\_sample + fraction \times (upper\_sample - lower\_sample)
$$

In this case, the $lower\_sample = -44$, $upper\_sample = -22$ and $fraction = 0.25$, as a result we have:

In [5]:
lower_sample = -44
upper_sample = -22
fraction = 0.25

q25 = lower_sample + fraction*(upper_sample - lower_sample)
q25

-38.5

In [6]:
test_array = np.array([-78, -56, -44, -22, -5, 12, 32, 45, 52, 61, 68, 78])
test_data = pd.DataFrame({
    'x' : test_array, 
})

plot_q25(test_data, [-44, -22, q25])

alt.LayerChart(...)

In [7]:
test_array = np.array([-78, -56, -44, -22, -5, 12, 32, 45, 52, 61, 68, 78])
test_data = pd.DataFrame({
    'x' : test_array, 
})


cut_point = -40.0
p1 = test_array[2]
p2 = test_array[3]

test_data['category'] = np.where(test_data['x'] < cut_point, 'Below 1st Quartile', 'Above 1st Quartile')


points = (alt
          .Chart(test_data)
          .mark_point(size = 220, 
                      filled=True,
                      stroke='black',
                      strokeWidth=1)
          .encode(
              x = alt.X('x'),
              color=alt.Color('category',
                                      scale=alt.Scale(domain=['Below 1st Quartile', 'Above 1st Quartile'],
                                                      range=['#17becf', '#f58518'])
                                     )
          )
         )

v1 = (alt
      .Chart(pd.DataFrame({'p1': [p1]}))
      .mark_rule(
          color = 'black',
          strokeWidth = 0.7
      )
      .encode(
          x = 'p1:Q',
          y = alt.value(2),
          y2 = alt.value(38)
      )
     )

a1 = (alt
      .Chart(test_data)
      .mark_text(
          align='center',
          baseline='top',
          fontSize = 12,
          dx = 0,
          dy = -30
      ).encode(
          x='x',
          text=alt.Text('x', format=',.0f')
      ).transform_filter(
          (datum.x == p1)
      )
     )

v2 = (alt
      .Chart(pd.DataFrame({'p2': [p2]}))
      .mark_rule(
          color = 'black',
          strokeWidth = 0.7
      )
      .encode(
          x = 'p2:Q',
          y = alt.value(2),
          y2 = alt.value(38)
      )
     )

a2 = (alt
      .Chart(test_data)
      .mark_text(
          align='center',
          baseline='top',
          fontSize = 12,
          dx = 0,
          dy = -30
      ).encode(
          x='x',
          text=alt.Text('x', format=',.0f')
      ).transform_filter(
          (datum.x == p2)
      )
     )

(points + v1 + v2 + a1 + a2).properties(
    width = 620,
    height = 40
)

alt.LayerChart(...)

In [8]:
test_data.iloc[2]['x']

-44

In [9]:
a1 = (alt
      .Chart(test_data)
      .mark_text()
      .encode(
          x='x',
          text=alt.Text('x', format='$,.0f')
      ).transform_filter(
          (datum.x == p1)
      )
     )
a1

alt.Chart(...)

In [10]:
p1

-44

At this stage, we start to understand why the definition of quantile does not give us a unique way to calculate it. We can set the first quantile to be the third value

### How to calculate the rank to use for the percentile

When calculating the rank to use fot the percentile, we need to make a decision about how we want to define the 0<sup>th</sup> and the 100<sup>th</sup>. We can define the 0<sup>th</sup> percentile as the minimum of the dataset and 100<sup>th</sup> as the maximum or assuming that these percentiles are not valid percentiles. From a statistical point of view, we can see each dataset as a sample drawn from a theoretically continuous distribution. In this case, what is the meaning of the 0<sup>th</sup> or 100<sup>th</sup> percentile?

For semplicity sake, let's consider a dataset containing 10 elements:

| Rank | Value |
|---:|---:|
| 1 | 2 |
| 2 | 5 |
| 3 | 9 |
| 4 | 14 |
| 5 | 21 |
| 6 | 23 |
| 7 | 35 |
| 8 | 47 |
| 9 | 47 |
| 10 | 57 |

or

|           |   |   |   |   |   |   |   |   |   |    |
|:----------|:--|:--|:--|:--|:--|:--|:--|:--|:--|:---|
| **Rank**  | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
| **Value** | 2 | 5 | 9 | 14 | 21 | 23 | 35 | 47 | 47 | 57 |

In [11]:
test_dataset = np.array([2, 5, 9, 14, 21, 23, 35, 47, 47, 57])
len(test_dataset)

10

In [15]:
n_of_elements = 10
percentile = 70
p = percentile/100
r_0 = p*n_of_elements
r_1 = p*(n_of_elements+1)
r_2 = p*(n_of_elements-1)

print(r_0)
print(r_1)
print(r_2)

7.0
7.699999999999999
6.3


In [16]:
plot_boxplot_with_points(df=sales,
                        obs='Sub-Category',
                         value='sales'
                        )

alt.LayerChart(...)

For the `sales` dataset, its `Q1` is $46674$, the `Median` is $114880$ and `Q3` is $203413$. In this case, we can say that the Copiers sales are higher than the $50\%$ of all the considered categories as its value, $149528$, is higher than the median. It is also less than `Q3`, meaning that it is not in the top $25\%$ with higher sales. As we can see, the notion of quantile, is based on ranking the data, in this case by sorting in ascending order.

In [17]:
sales[sales['Sub-Category'] == 'Copiers']

,Sub-Category,sales,type
6,Copiers,149528.0,Sub-Category


In the second one, without the median point, we have:
- `Q1` = 41784.85
- `Q3` = 204300.93

So the idea is to sort the dataset and split its data in four evenly parts evenly. The first quartile indicate the value for which the $25\%$ of the data are below this point. The second quartile is the median, the third quartile contains $75\%$ of the data. As we will see in the next sections, first and third quartile will define the intequartile range, pivotal in spotting possible outliers of the dataset.

In calculating the first and third quartile, we follow a similar approach as for the median. We compute the index of the first $1/4$ of the data ($3/4$ for the third quartile), we check if this is an odd or even number and split the data accordingly, performing a linear interpolation if needed. As before, I will use the `sales` and `sales_even` dataset to illustrate these points.

Let's start with the `sales` dataset which contains an odd number (17) of elements. In this case, as we previously discussed, the median is the middle value of the sorted dataset, i.e. its 9<sup>th</sup> element.

In [18]:
test_dataset = np.array([2,4,6,8,13, 16, 22, 35, 40, 42, 48])
len(test_dataset)

11

In [19]:
p=0.7
p*(len(test_dataset)-1)

7.0

In [20]:
test_dataset[int(p*(len(test_dataset)-1))]

35

In [21]:
np.quantile(test_dataset, q = 0.7)

35.0

In [22]:
0.7 * 9

6.3

In [23]:
50*0.65

32.5

In [24]:
0.25*9

2.25

In [25]:
0.25*2

0.5

In [26]:
test_dataset = np.array([3,5,7,8,9,11,13,15])
np.percentile(test_dataset, 25)

6.5

In [27]:
0.7*12

8.399999999999999

In [28]:
35+(0.4*5)

37.0

In [ ]:
np.percentile(test_dataset, 70)

In [ ]:
test_dataset = np.array([33,48,57,65,67, 69, 75, 77, 77, 78, 80,
                        83, 85,86,87,88,89,90,99,99,99,99,100,101,105])
len(test_dataset)

In [ ]:
np.percentile(test_dataset,20)

In [ ]:
mquintile(test_dataset, 0.25)

In [ ]:
0.2*24

In [ ]:
plot_boxplot_with_points(sales, 'Sub-Category', 'sales')

In [ ]:
p = 0.5
n25 = p*(len(sales['sales'])-1)
n25

In [ ]:
5/17

In [ ]:
np.median(np.arange(1,101))

In [ ]:
np.percentile(np.arange(1,101), 0.5)

In [ ]:
mquintile(np.arange(1,101), 0.5)

In [ ]:
def dataset_median(dataset):
    """
    Compute the median of a dataset using linear interpolation

    Parameters:
    -----------
    dataset : array-like
        Array of observations (pandas Series)
    
    Returns:
    --------
    float
        The computed quantile value
    """
    
    midpoint = len(dataset)/2
    
    if midpoint%2 != 0:
        # if the lenths of the dataset is odd, return the middle point 
        return(dataset.sort_values().iloc[int(midpoint)])
    # otherwise, return the arithmetic mean of the two middle points
    return((dataset.sort_values().iloc[int(midpoint)-1]+dataset.sort_values().iloc[int(midpoint)])/2)

If we consider the first quartile, i.e. $25\%$ of the sorted data, we start by calculating the number of elements that would be contained in the quartile, i.e. divide the size of the dataset by $4$. This number will also be the index of the maximum element of the quartile if we sort the data in ascending order and so, by definition, it is the value of the quartile. Similarly as the the median, dividing the size of the dataset in 4 can result into a fraction or a whole number. If the result is a whole number, the element at that position will be the median. If it is not a whole number, we have to interpolate between the two adjacent points between this theoretical position. To interpolate, we consider the fraction of the theoretical position and return:

$$
q25 = lower\_sample + fraction \times (upper\_sample - lower\_sample)
$$

The following, is a function computing the first quartile based on what we just described.

In [ ]:
def q25(data):

    """
    data: np array, list, pandas series is an array of observations
    """

    # p is the percentage of the data in the first quartile, i.e 25%
    p = 0.25
    samples = np.sort(data)
    # n is the position of the sorted array containing the samples in the desired quantile
    n = p*(len(samples)-1)
    print("theoretical position of the first quartile: {}".format(n))
    if n%1 == 0:
        # if the position is a whole number, return the sample at that position 
        print("Whole number")
        print(samples[int(n)])
        return(samples[int(n)])
    else:
        # is the position is an odd number, we compute the the values of the sorted array
        # for the considered position
        pos = int(n)
        # compute the adiacent samples to interpole to compute the quartile
        lower_sample = samples[pos]
        upper_sample = samples[pos+1]
        print("lower sample = {}, upper sample {}".format(lower_sample, upper_sample))
        # compute the fraction of sample to use in the interpolation
        f = n-pos
        print("fraction = {}".format(f))
        # Finally, calculate the interpolated point representing the quantile
        quantile = lower_sample+(f * (upper_sample-lower_sample))
        print("quantile value = {}".format(quantile))
        return(quantile)

In [ ]:
q25(sales['sales'])

In [ ]:
q25(sales_even['sales'])

In [ ]:
p = 0.25
n25 = p*(len(sales['sales'])-1)
n25

As for the third quartile, we have the same procedure by considering the $75\%$ of the data as a proportion. We can now write a general function for each percentile of the dataset implementing the following algorithm:

1. Sort all numbers in the dataset from smallest to largest
2. Calculate the position (n) where your desired quantile would be: 
    - Multiply the quantile (p) by (number of samples - 1)
3.	Look at this position number (n): 
    - If it's a whole number (like 2.0): 
        - Your answer is simply the value at that position
    - If it's a decimal number (like 2.3): 
        - Take the values at the position before and after the decimal
        - Calculate how far between these two values you should go (using the decimal part)
        - Use this to calculate a weighted average of the two values


In [ ]:
q25_sales = sales.sort_values('sales').iloc[int(n25)]['sales']

In [ ]:
sales['sales'].quantile(0.25)

In [ ]:
import string
string.ascii_uppercase[:10]

In [ ]:
dataset_size = 4
np.random.seed(100)
x = np.round(100*np.random.normal(size = dataset_size),1)
cat = list(string.ascii_uppercase[:dataset_size])

df_test = pd.DataFrame(
    {
        'x' : x,
        'cat' : cat
    }
)

n25 = 0.25*(len(df_test['x'])-1)
n25

df_test.sort_values('x')

In [ ]:
# plot_boxplot_with_points(df, obs, value, show_axis = True, title = "", show_title = True):
plot_boxplot_with_points(df_test, 'cat', 'x')

In [ ]:
mquintile(df_test['x'], 0.25)

In [ ]:
print(np.quantile(df_test['x'], 0.25))

In [ ]:
sales.head()

In [ ]:
from altair import datum #Needed for subsetting (transforming data)

points = (alt
          .Chart(sales)
          .mark_point(size = 50, 
                      filled=True,
                      stroke='black',
                      strokeWidth=1)
          .encode(
              x = alt.X('sales:Q'),
              color = alt.condition(
                  alt.datum.sales < q25_sales,
                  alt.value("#17becf"),
                  alt.value("#f58518")
              ),
              tooltip = ['Sub-Category:N', 'sales:Q']                        
          )
         )



box = (alt
       .Chart(sales)
       .mark_boxplot(size = 25, 
                     opacity = 0.8, 
                     color = '#76B7B2')
       .encode(
           x = alt.X('sales:Q'),
       )
      )
       

chart = (box   + points).properties(
    title = 'Sales by Sub-category box plot',
    width = 620,
    height = 80
)

chart

In [ ]:
def mquintile(data, p):
    """
    data: np array, list, pandas series is an array of observations
    p: float between 0 and 1, is the percentage of samples you want to consider
    """

    samples = np.sort(data)
    # n is the position of the sorted array containing the samples in the desired quantile
    n = p*(len(samples)-1)
    if n%1 == 0:
        # if the position is a whole number, return the sample at that position 
        print("Whole number")
        print(samples[int(n)])
        return(samples[int(n)])
    else:
        # is the position is an odd number, we compute the the values of the sorted array
        # for the considered position
        pos = int(n)
        # compute the adiacent samples to interpole to compute the quartile
        lower_sample = samples[pos]
        upper_sample = samples[pos+1]
        print("lower sample = {}, upper sample {}".format(lower_sample, upper_sample))
        # compute the fraction of sample to use in the interpolation
        f = n-pos
        print("fraction = {}".format(f))
        # Finally, calculate the interpolated point representing the quantile
        quantile = lower_sample+(f * (upper_sample-lower_sample))
        print("quantile value = {}".format(quantile))
        return(quantile)

In [ ]:
mquintile(sales_even['sales'], 0.25)

In [ ]:
np.quantile(sales_even['sales'], 0.25)

In [ ]:
N = 16
p = 0.25
p*(N-1)

In [ ]:
p1 = plot_boxplot_with_points(sales, 'Sub-Category', 'sales', show_axis=False, title = "Sales by Subcategories - odd and even number of elements")
p2 = plot_boxplot_with_points(sales_even, 'Sub-Category', 'sales')

(p1 & p2).resolve_scale(
    x='shared'
)

In [ ]:
2

In [ ]:
test_dataset = np.array([2,4,6,8,13, 16, 22, 35, 40, 42, 48])
len(test_dataset)

In [ ]:
np.quantile(test_dataset, 0.7)

In [ ]:
0.7*11

In [ ]:
4/11

In [ ]:
3/11

In [ ]:
100*(8-0.5)/11

## Reference

[Percentiles: Interpretations and Calculations](https://statisticsbyjim.com/basics/percentiles/)

In [ ]:
test = np.sort(np.round(100 * np.random.random(size = 7)))
df_test = pd.DataFrame(
    {
        'x' : test 
    }
)

In [ ]:
df_test

In [ ]:
len(test)%4

In [ ]:
point = (alt
         .Chart(df_test)
         .mark_point(size = 50, 
                      filled=True,
                      stroke='black',
                      strokeWidth=1)
         .encode(
             x = 'x'
         )    
)

box = (alt
 .Chart(df_test)
 .mark_boxplot()
 .encode(
     x = 'x'
 )
)

box + point

In [ ]:
11%4

In [ ]:
test = np.array([33, 34, 55, 57, 60, 61, 61, 2, 3, 5, 6, 7, 8, 12, 15, 18, 20, 27, 28, 29, 70])
len(test)

In [ ]:
len(test)%4

In [ ]:
df_test = pd.DataFrame(
    {
        'x' : test 
    }
)

In [ ]:
mquintile(df_test['x'], p = 0.25)

In [ ]:
np.sort(test)

In [ ]:
21/4

In [ ]:
16/len(test)

In [ ]:
(16*6*20)/(25*30)

In [ ]:
64/25

In [ ]:
test = np.array([55, 60, 65, 70, 75, 80, 85, 90, 95, 100])

In [ ]:
np.percentile(test, 70)

In [ ]:
len(test)

In [ ]:
np.percentile(np.array([3, 5, 7, 8, 9, 11, 13, 15]), 25)

In [ ]:
np.quantile(test_dataset, 0.3)

## Excel vs Python: N+1 or N-1? This is the question.

> In calculating the percentile of a discrete dataset, I found that same definitions use r = p(N+1), where r is the rank, p is the percentile divided by 100 and N the total number of samples. Other definitions uses r = p(N-1), like linear interpolation in numpy.percentile. Why there is this difference and which one is better?

Let me help explain this difference in percentile calculation methods. The key difference between these methods lies in how they handle the extreme values (0th and 100th percentiles) and their underlying assumptions:

- $r = p \times (N+1)$ method:

This is sometimes called the Excel method and 
- places the 0th percentile below the minimum value;
- places the 100th percentile above the maximum value
- Assumes the data points represent the middle of each portion

Good when you want to be explicit that there could be values outside your sample range

- $r = p*(N-1)$ method:


Used by NumPy and other statistical software
- Places the 0th percentile at the minimum value
- Places the 100th percentile at the maximum value
- Assumes the data points represent the boundaries of portions
More appropriate when you believe your sample contains the true extremes

Example:
Consider a sorted dataset [10, 20, 30, 40, 50] (N=5)
For the 25th percentile (p=0.25):

(N+1) method: r = 0.25*(5+1) = 1.5
(N-1) method: r = 0.25*(5-1) = 1.0

Neither method is universally "better" - it depends on your assumptions about the data and your specific needs. The (N-1) method is often preferred in scientific computing because:

- It's more conservative about extrapolating beyond observed values
- It ensures the minimum and maximum values are actually achievable
- It's more mathematically consistent with continuous distribution theory

If you're working with a large dataset where the distinction between these methods becomes minimal, the choice matters less. However, for small datasets or when extreme values are important, you should choose based on whether you believe your sample contains the true population extremes. Would you like me to elaborate on any of these points or show how they differ in specific calculations?

The concept of percentile is apparently clear but, in itself, has some ambiguities that, when we try to calculate analytically, brings to different methods about how to do it. This different methods produce same results when dealing with large datasets while may show different results with smaller ones. 

## Excel `PERCENTILE.INC()` and `PERCENTILE.EXC()`

- `PERCENTILE.EXC()` $N+1$

In [ ]:
def mpercentile_exc(lst, percentile):
    lst = np.sort(lst)
    p = percentile/100
    n = len(lst)
    # calculate the rank to use for the percentile
    r = p*(n+1)
    print("rank excluding: {}".format(r))
    ind = int(r)
    if (r%1) == 0.0:
        return(lst[ind-1])
    return(lst[ind-1]+(lst[ind] - lst[ind-1])*(r%1))

- `PERCENTILE.INC()` $N-1$

In [ ]:
def mpercentile_inc(lst, percentile):
    lst = np.sort(lst)
    p = percentile/100
    n = len(lst)
    # calculate the rank to use for the percentile
    r = p*(n-1)
    print("rank including: {}".format(r))
    ind = int(r)
    if (r%1) == 0.0:
        return(lst[ind])
    return(lst[ind]+(lst[ind+1] - lst[ind])*(r%1))

In [ ]:
def mpercentile_pn(lst, percentile):
    lst = np.sort(lst)
    p = percentile/100
    n = len(lst)
    # calculate the rank to use for the percentile
    r = p*(n)
    print("rank including: {}".format(r))
    ind = int(r)
    if (r%1) == 0.0:
        return(lst[ind])
    return(lst[ind]+(lst[ind+1] - lst[ind])*(r%1))

In [ ]:
test_dataset = np.array([2,35,6,8,40, 16, 22, 4, 13, 42, 48])
perc = 90

print("Include: {}".format(np.round(mpercentile_inc(test_dataset, perc),2)))
print("N: {}".format(np.round(mpercentile_pn(test_dataset, perc),2)))
print("Numpy:   {}".format(np.quantile(test_dataset, perc/100)))
print("Exclude: {}".format(np.round(mpercentile_exc(test_dataset, perc),2)))

In [ ]:
test_dataset = np.array([2,35,6,8,40, 16, 22, 4, 13, 42])
perc = 0

print("Include: {}".format(np.round(mpercentile_inc(test_dataset, perc),2)))
print("N: {}".format(np.round(mpercentile_pn(test_dataset, perc),2)))
print("Numpy:   {}".format(np.quantile(test_dataset, perc/100)))
print("Exclude: {}".format(np.round(mpercentile_exc(test_dataset, perc),2)))

In [ ]:
test_dataset = np.array([2,2,35,6,8,40, 16, 22, 4, 13, 42, 48,48])
perc = 10

print("Include: {}".format(np.round(mpercentile_inc(test_dataset, perc),2)))
print("Numpy:   {}".format(np.quantile(test_dataset, perc/100)))
print("Exclude: {}".format(np.round(mpercentile_exc(test_dataset, perc),2)))

In [ ]:
np.random.seed(10)
n_elements = 19
percentile = 70
test_dataset = np.round(500*np.random.uniform(size = n_elements),0)
test_dataset

In [ ]:
test_dataset = np.array([3,4,4,6,7,9,12, 13, 14,16,17,19,22,23,23,25,28,29,34,37])
perc = 100

print("Include: {}".format(np.round(mpercentile_inc(test_dataset, perc),2)))
print("Numpy:   {}".format(np.quantile(test_dataset, perc/100)))
print("Exclude: {}".format(np.round(mpercentile_exc(test_dataset, perc),2)))

The reason you can't simply use $rank = p \times N$ to compute percentiles in Excel (or in general) comes down to how percentiles are defined in terms of the rank and how the percentiles should relate to the data points themselves. The formulas you mentioned—PERCENTILE.EXC() and PERCENTILE.INC()—are designed to fit different definitions of percentiles that have distinct behaviors when dealing with the boundaries of the dataset (i.e., the minimum and maximum values). Let’s break down why the approach of rank = p * N doesn't work for both cases:

#### 1. Why not just rank = p * N?

In theory, you could compute a rank as p * N for a given percentile. This would give you a position in the data, but it doesn't account for whether that position aligns with the desired percentile definition.
- For PERCENTILE.EXC(): The logic behind the rank = p * (N + 1) formula is that you're working with a continuous distribution and excluding the extreme percentiles, meaning that you don't consider the actual minimum and maximum values of the dataset as valid percentiles. Therefore, the rank calculation needs to be scaled slightly beyond the bounds of the actual data points (using N + 1 instead of N). This makes the rank calculation reflect the fact that the extreme percentiles are excluded from the dataset.
- For PERCENTILE.INC(): Conversely, in the case of the PERCENTILE.INC() function, you're including the actual minimum and maximum values as part of the dataset. This means that the rank should be computed as p * (N - 1) because the inclusion of the extremes makes the rank distribution more "compact" — the boundary values themselves are counted in the computation of percentiles. The rank formula p * (N - 1) ensures that the resulting rank stays within the actual bounds of the data.

#### 2. What would happen with rank = p * N?

If you used the formula rank = p * N in both cases, you’d run into two issues:
- For PERCENTILE.EXC(): With rank = p * N, you would get a rank that assumes the minimum and maximum values can be part of the range for percentiles, even though in the EXC method, those values should not be considered part of the percentiles. The rank value could therefore end up being outside the range of valid data positions, which would cause misalignment with the intended interpretation of percentiles.
For example, if you have a dataset with 5 values and want to calculate the 90th percentile, using rank = 0.9 * 5 = 4.5 suggests an interpolated rank between values 4 and 5, but if you were to use rank = 0.9 * (5+1) = 5.4, you are explicitly saying the rank could be between the 5th and 6th data points (which doesn't exist), and hence the percentile would lie outside the bounds.
- For PERCENTILE.INC(): The formula rank = p * N would imply that you're treating the extreme percentiles (the 0th and 100th percentiles) as being outside the dataset, which contradicts the goal of PERCENTILE.INC(), where both the minimum and maximum values are part of the percentile calculation. In this case, the correct rank needs to be adjusted to include those boundary points in the calculation, which is why the formula uses p * (N - 1) instead of p * N.

#### 3. What’s the significance of N + 1 vs. N - 1?
- PERCENTILE.EXC() (using N + 1): The formula p * (N + 1) results from the idea that the minimum and maximum values are excluded from the 0th and 100th percentiles in an exclusive method. The rank needs to reflect the fact that the dataset is considered as part of a larger "continuum" where the extreme values are not explicitly part of the dataset's percentiles.
- PERCENTILE.INC() (using N - 1): The formula p * (N - 1) ensures that the 0th and 100th percentiles are counted as part of the dataset, with the rank values corresponding directly to positions within the actual data range. This inclusion reflects the idea that the percentiles should be distributed evenly across the actual data points.

####  Summary
The formulas rank = p * (N + 1) and rank = p * (N - 1) are used instead of rank = p * N because they respect the distinction between whether you’re including or excluding the boundary values in the percentile calculation. Using rank = p * N would fail to account for the correct handling of boundary values in either method. Thus, each approach (exclusive vs inclusive) requires an adjustment in how the rank is calculated to ensure the percentiles align with their intended definitions


In [ ]:
0.7*

In [ ]:
print(1.1 * 3)

In [ ]:
import decimal
from decimal import Decimal


x = Decimal('0.1')
y = Decimal('0.1')
z = Decimal('0.1')

s = x + y + z

print(s)

## Quartile

In [ ]:
test_array = np.array([-78, -56, -44, -22, -5, 12, 32, 45, 52, 61, 68, 78])
test_data = pd.DataFrame({
    'x' : test_array, 
})

cut_point = -40.0

points = (alt
          .Chart(test_data)
          .mark_point(size = 220, 
                      filled=True,
                      stroke='black',
                      strokeWidth=1)
          .encode(
              x = alt.X('x'),
              color = alt.condition(
                  alt.datum.x < cut_point,
                  alt.value("#17becf"),
                  alt.value("#f58518")
              )
          )
         )

points.properties(
    width = 620,
    height = 40
)

Let's start with the first quartile. It is defined as:

1. the smallest value that is greater than the $25/%$ of the data
2. Definition 2: the smallest value that is greater or equal than the $25/%$ of the data
3. Definition 3: An interpolated value between the two closest rank

If I want to use Definition 3, how do I compute the rank? And how I interpolate between these two points? Intuitively, if I have a sorted dataset with 12 elements, the first quartile will be greater than the first three elements so that $1/4$ (i.e. $3/12$) of the data lies below this cut point. 

|           |   |   |   |   |   |   |   |   |   |    |    |    |
|:----------|:--|:--|:--|:--|:--|:--|:--|:--|:--|:---|:---|:---|
| **Rank**  | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 | 12 |
| **Value** | $a_{1}$ | $a_{2}$ | $a_{3}$ | $a_{4}$ | $a_{5}$ | $a_{6}$ | $a_{7}$ | $a_{8}$ | $a_{9}$ | $a_{10}$ | $a_{11}$ | $a_{12}$ |



In [ ]:
point = (alt
         .Chart(test_data)
         .mark_point(size = 50, filled=True, stroke='black',
                     strokeWidth=1,color = '#9C755F')
         .encode(
             x = alt.X('x', axis=alt.Axis(labels=False, title=None, grid=True)),
             tooltip = ['x']
         )
        )
point

In [ ]:
test_data.sort_values('x')

In [ ]:
box = (alt
       .Chart(test_data)
       .mark_boxplot(size = 25, opacity = 0.8, color = '#76B7B2')
       .encode(
           x = 'x',
       )
      )


point = (alt
         .Chart(test_data)
         .mark_point(size = 50, filled=True, stroke='black',
                     strokeWidth=1,color = '#9C755F')
         .encode(
             x = alt.X('x', axis=alt.Axis(labels=False, title=None, grid=True)),
             tooltip = ['x']
         )
        )

(box + point).properties(
    width = 620,
    height = 80
)

In [ ]:
test_data.sort_values('x')